# LTSR Colab Phase 7 — DREAM4

Size10/100をseed×network shardで実行し、trajectory leakageを防ぐ。

実行順は、Python 3.10確認 → Drive mount/source固定 → Phaseセルである。
Python環境導入でruntimeが再起動した場合は、再接続して最初のセルからやり直す。

In [ ]:
# 必ず最初に実行する。Colab UI kernelとは別に研究コード用Python 3.10を用意する。
import subprocess
import sys
from pathlib import Path

PY310 = Path("/usr/local/bin/python")

def worker_version():
    if not PY310.is_file():
        return None
    return subprocess.check_output(
        [
            str(PY310), "-c",
            "import sys; print('.'.join(map(str, sys.version_info[:3])))",
        ],
        text=True,
    ).strip()

version = worker_version()
print("Colab controller:", sys.version)
print("LTSR worker before setup:", version)
if version is None or not version.startswith("3.10."):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.10"
    ])
    import condacolab
    condacolab.install_miniconda()
    raise SystemExit(
        "Python 3.10 workerを導入しました。runtime再起動後、このセルを再実行してください。"
    )
print("LTSR worker Python 3.10: OK —", version)

In [ ]:
# Google認証とDrive mountはユーザー自身が行う。
from google.colab import drive
drive.mount("/content/drive")

import json
import os
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/LTSR_colab")
REPO_ROOT = Path("/content/LTSR")
BRANCH = "20260726/gpu-scale-prep-colab"
REPO_URL = (
    "https://github.com/blabo25226/"
    "Layer-selective_Transformer-based_Symbolic_Regression.git"
)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
lock_path = DRIVE_ROOT / "source_lock.json"

if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )

if lock_path.is_file():
    locked_commit = json.loads(lock_path.read_text(encoding="utf-8"))["commit"]
    subprocess.run(["git", "checkout", "--detach", locked_commit], cwd=REPO_ROOT, check=True)
else:
    locked_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True
    ).strip()
    partial = lock_path.with_suffix(".json.partial")
    partial.write_text(
        json.dumps({"branch": BRANCH, "commit": locked_commit}, indent=2),
        encoding="utf-8",
    )
    os.replace(partial, lock_path)

sys.path.insert(0, str(REPO_ROOT / "src"))
from colab_runtime import assert_locked_source, require_python_310

require_python_310(PY310)
print("locked commit:", assert_locked_source(REPO_ROOT, DRIVE_ROOT))
print("Drive root:", DRIVE_ROOT)

In [ ]:
# Phase 4--8で同じ3値を使用する。別設定は必ず新しいRUN_IDにする。
from colab_runtime import config_for

RUN_KIND = "smoke"  # "smoke" -> "pilot" -> "paper"
RUN_ID = "colab_smoke_20260726_01"
MAX_PARALLEL_SEEDS = 1  # smoke実測後にのみ増やす
CONFIG = config_for(
    RUN_KIND, RUN_ID, max_parallel_seeds=MAX_PARALLEL_SEEDS
)
print(json.dumps(CONFIG.scientific_dict(), indent=2, ensure_ascii=False))

## Phase 7を実行

同じrunの前Phase成果物をDriveから復元し、`START_PHASE=7`、
`STOP_AFTER_PHASE=7`、`STRICT_RESUME=1`で実行する。
3分ごと、および終了・例外時にDriveへ同期する。

In [ ]:
from colab_runtime import run_phase

output = run_phase(
    REPO_ROOT, DRIVE_ROOT, CONFIG, 7,
    python_executable=PY310,
)
print("Phase 7 output:", output)

In [ ]:
payload = json.loads(output.read_text(encoding="utf-8"))
print("top-level keys:", sorted(payload) if isinstance(payload, dict) else type(payload))
print("Drive checkpoint:", DRIVE_ROOT / "runs" / RUN_ID)